<a href="https://colab.research.google.com/github/sila85/production_ml_demo/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessed tokens from Module 2
tokens = [
    'love', 'natur', 'languag', 'process', 'excit',
    'field', 'nlp', 'help', 'comp ut', 'understand',
    'human', 'languag', 'differ', 'applic', 'nlp',
    'includ', 't ranslat', 'chatbot', 'sentiment',
    'analysi'
]

# Correct accidental spaces in tokens
tokens = [word.replace(" ", "") for word in tokens]

# Convert tokens into a sentence
sentence = " ".join(tokens)

# Create a dataset
df = pd.DataFrame({
    "preprocessed_text": [sentence]
})

print(df)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer()
X_bow = bow_vectorizer.fit_transform(df["preprocessed_text"])

bow_features = bow_vectorizer.get_feature_names_out()

print("BoW vocabulary:")
print(bow_features)

print("\nBoW matrix:")
print(X_bow.toarray())

print("\nBoW matrix shape:", X_bow.shape)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df["preprocessed_text"])

tfidf_features = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF vocabulary:")
print(tfidf_features)

print("\nTF-IDF matrix:")
print(np.round(X_tfidf.toarray(), 3))

print("\nTF-IDF matrix shape:", X_tfidf.shape)

In [ ]:
%pip install gensim

In [ ]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-100")

print("Embedding dimensions:", glove.vector_size)
print("Vector for 'love':", glove["love"][:10])

In [ ]:
def sentence_embedding(tokens, model):
    vectors = [
        model[word]
        for word in tokens
        if word in model
    ]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

glove_vector = sentence_embedding(tokens, glove)

print("GloVe vector shape:", glove_vector.shape)
print("First 10 features:", glove_vector[:10])

In [ ]:
bow_counts = np.asarray(X_bow.sum(axis=0)).flatten()

bow_df = pd.DataFrame({
    "Word": bow_features,
    "Frequency": bow_counts
}).sort_values("Frequency", ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=bow_df, x="Frequency", y="Word")
plt.title("Top Words by Bag-of-Words Frequency")
plt.xlabel("Frequency")
plt.ylabel("Word")
plt.tight_layout()
plt.show()

In [ ]:
tfidf_weights = X_tfidf.toarray()[0]

tfidf_df = pd.DataFrame({
    "Word": tfidf_features,
    "TF-IDF Weight": tfidf_weights
}).sort_values("TF-IDF Weight", ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=tfidf_df, x="TF-IDF Weight", y="Word")
plt.title("TF-IDF Word Weights")
plt.xlabel("TF-IDF Weight")
plt.ylabel("Word")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA

valid_tokens = [word for word in tokens if word in glove]

if len(valid_tokens) >= 2:
    word_vectors = np.array([glove[word] for word in valid_tokens])

    pca = PCA(n_components=2)
    word_vectors_2d = pca.fit_transform(word_vectors)

    plt.figure(figsize=(10, 7))
    plt.scatter(word_vectors_2d[:, 0], word_vectors_2d[:, 1])

    for i, word in enumerate(valid_tokens):
        plt.annotate(
            word,
            (word_vectors_2d[i, 0], word_vectors_2d[i, 1])
        )

    plt.title("GloVe Word Embeddings (PCA)")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough words matched the GloVe vocabulary.")

In [ ]:
comparison = pd.DataFrame({
    "Method": ["BoW", "TF-IDF", "GloVe"],
    "Feature Dimensions": [
        X_bow.shape[1],
        X_tfidf.shape[1],
        glove_vector.shape[0]
    ],
    "Representation": [
        "Word counts",
        "Weighted word values",
        "Dense semantic vectors"
    ]
})

print(comparison.to_string(index=False))